# Rule: **build_hourly_heat_demand**

**Description**

This rule builds proxies for hourly heat demand time series. Profiles are given for:

(`water`, `space`) $\times$ (`residential`, `services`)

Intraday profiles are from BDEW (includes weekdays/weekend).

For **space**, the daily heat demand is multiplied by the corresponding intraday profile to obtain the hourly heat demand time series. 

For **water**, only the corresponding intraday profile is employed.

**Outputs**

- resources/`hourly_heat_demand_total_base_s_{clusters}.nc`
- resources/`residential_heat_dsm_profile_total_base_s_{clusters}.csv`

In [ ]:
######################################## Parameters

### Run
name = 'case_SectorCoupled'
prefix = ''

### Network
clusters = 5

In [ ]:
##### Import packages
import xarray as xr
import os
import sys
import matplotlib.pyplot as plt
import pandas as pd

##### Import local functions
sys.path.append(os.path.abspath(os.path.join('..')))
import functions as xp


##### Read params.yaml
params = xp.read_params('../params.yaml')


##### Ignore warnings
import warnings
warnings.filterwarnings('ignore', category=UserWarning)

## `hourly_heat_demand_total_base_s_{clusters}.nc`

Load the dataset and show its structure.

In [ ]:
file = f'hourly_heat_demand_total_base_s_{clusters}.nc'
path = f'{params["rootpath"]}/resources/{prefix}/{name}/'

ds = xr.open_dataset(path + file)

ds

Plot total hourly heat demand aggregated over nodes for all variables in the dataset.

In [ ]:
#################### Parameters
date_start = '2013-01-01'
date_end   = '2013-01-31'



#################### Plot
time_dim = 'snapshots'
node_dim = 'node'

vars_space = [v for v in ds.data_vars if 'space' in v]
vars_water = [v for v in ds.data_vars if 'water' in v]

fig, (ax_space, ax_water) = plt.subplots(2, 1, figsize=(10, 8))

for ax, group_vars, group_label in [
    (ax_space, vars_space, 'space'),
    (ax_water, vars_water, 'water'),
]:
    for v in group_vars:
        da = ds[v]
        if node_dim in da.dims:
            series = da.sum(dim=node_dim).to_series()
        else:
            series = da.to_series()
        series = series.sort_index().loc[date_start:date_end]
        series.plot(ax=ax, label=v, linewidth=0.8, alpha=0.85)
    ax.set_title(f'Hourly heat demand — {group_label} (aggregated over nodes)')
    ax.set_xlabel('Time')
    ax.set_ylabel('[-]')
    ax.grid(True, linestyle='--', alpha=0.4)
    ax.legend(title='Variable', ncol=1, fontsize=8)

plt.tight_layout()

## `residential_heat_dsm_profile_total_base_s_{clusters}.csv`

Load the CSV profile and show its structure.

In [ ]:
file = f'residential_heat_dsm_profile_total_base_s_{clusters}.csv'
df_dsm = xp.load_file_csv(
    params,
    filename=file,
    prefix=prefix,
    name=name,
    location='resources',
    index_col=0,
)

df_dsm.head()